# Estudo de Simulação — Execução por Cenário

Este notebook organiza a análise por cenário. Para cada cenário temos: 1) Definição do cenário; 2) Funções (uso das funções comuns); 3) Métricas; 4) Resultados; 5) Análises.

Execute as células em sequência.

## Dependências
Caso esteja no Colab, descomente a instalação abaixo.

In [ ]:
# !pip -q install numpy pandas scipy statsmodels matplotlib

## Imports e configuração comum

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

@dataclass(frozen=True)
class StudyConfig:
    seed: int = 20260507
    replications: int = 1000
    sample_sizes: tuple = (30, 100)
    alpha: float = 0.05
    beta0: float = 1.0
    beta1: float = 2.0
    sigma: float = 1.0
    x_low: float = -2.0
    x_high: float = 2.0
    t_df: int = 3
    gamma_shape: float = 2.0
    gamma_scale: float = 1.0
    delta_const: float = 1.0
    delta_x: float = 0.6
    ar1_rho: float = 0.7
    lognormal_sigma: float = 0.4

SCENARIO_LABELS = {
    'classic_normal': 'C1_classico_normal',
    'heavy_tails': 'C2_caudas_pesadas',
    'skewed_errors': 'C3_erros_assimetricos',
    'nonzero_mean_const': 'C4a_media_erro_nao_nula_constante',
    'nonzero_mean_x_dep': 'C4b_media_erro_dependente_X',
    'ar1_errors': 'C5_erros_correlacionados_ar1',
    'exponential_lognormal': 'C6_relacao_exponencial_lognormal',
}
SCENARIOS = list(SCENARIO_LABELS.keys())
cfg = StudyConfig()
ROOT = Path.cwd().parent
OUTPUTS = ROOT / 'outputs'
TABLES = OUTPUTS / 'tables'
FIGURES = OUTPUTS / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(cfg.seed)

## Funções comuns (geração, ajuste e métricas)

In [ ]:
def generate_x(n, rng, cfg):
    return rng.uniform(cfg.x_low, cfg.x_high, size=n)

def _normal_errors(n, rng, sigma):
    return rng.normal(0.0, sigma, size=n)

def _heavy_tail_errors(n, rng, cfg):
    base = rng.standard_t(df=cfg.t_df, size=n)
    scale = cfg.sigma / np.sqrt(cfg.t_df / (cfg.t_df - 2))
    return base * scale

def _skewed_errors(n, rng, cfg):
    g = rng.gamma(shape=cfg.gamma_shape, scale=cfg.gamma_scale, size=n)
    g_centered = g - (cfg.gamma_shape * cfg.gamma_scale)
    g_sd = np.sqrt(cfg.gamma_shape * (cfg.gamma_scale**2))
    return (g_centered / g_sd) * cfg.sigma

def _ar1_errors(n, rng, cfg):
    rho = cfg.ar1_rho
    innovations_sd = cfg.sigma * np.sqrt(1 - rho**2)
    u = rng.normal(0.0, innovations_sd, size=n)
    eps = np.zeros(n)
    eps[0] = rng.normal(0.0, cfg.sigma)
    for i in range(1, n):
        eps[i] = rho * eps[i - 1] + u[i]
    return eps

def generate_dataset(scenario, n, rng, cfg, beta0, beta1):
    x = generate_x(n, rng, cfg)
    if scenario == 'classic_normal':
        eps = _normal_errors(n, rng, cfg.sigma)
        y = beta0 + beta1 * x + eps
    elif scenario == 'heavy_tails':
        eps = _heavy_tail_errors(n, rng, cfg)
        y = beta0 + beta1 * x + eps
    elif scenario == 'skewed_errors':
        eps = _skewed_errors(n, rng, cfg)
        y = beta0 + beta1 * x + eps
    elif scenario == 'nonzero_mean_const':
        eps = cfg.delta_const + _normal_errors(n, rng, cfg.sigma)
        y = beta0 + beta1 * x + eps
    elif scenario == 'nonzero_mean_x_dep':
        eps = cfg.delta_x * x + _normal_errors(n, rng, cfg.sigma)
        y = beta0 + beta1 * x + eps
    elif scenario == 'ar1_errors':
        eps = _ar1_errors(n, rng, cfg)
        y = beta0 + beta1 * x + eps
    elif scenario == 'exponential_lognormal':
        eps = rng.normal(0.0, cfg.lognormal_sigma, size=n)
        eta = np.exp(eps)
        y = np.exp(beta0 + beta1 * x) * eta
    else:
        raise ValueError('Cenario invalido')
    return x, y

def fit_ols(x, y, alpha):
    X = sm.add_constant(x, has_constant='add')
    model = sm.OLS(y, X).fit()
    ci = model.conf_int(alpha=alpha)
    return {
        'beta0_hat': float(model.params[0]),
        'beta1_hat': float(model.params[1]),
        'ci_beta0_low': float(ci[0, 0]),
        'ci_beta0_high': float(ci[0, 1]),
        'ci_beta1_low': float(ci[1, 0]),
        'ci_beta1_high': float(ci[1, 1]),
        'pvalue_beta1': float(model.pvalues[1]),
        'fitted': np.asarray(model.fittedvalues),
        'resid': np.asarray(model.resid),
    }

def summarize_parameter(estimates, true_value):
    mean_hat = float(np.mean(estimates))
    bias = mean_hat - true_value
    variance = float(np.var(estimates, ddof=1))
    mse = float(np.mean((estimates - true_value) ** 2))
    return mean_hat, bias, variance, mse

def coverage_rate(ci_low, ci_high, true_value):
    return float(np.mean((ci_low <= true_value) & (true_value <= ci_high)))

def rejection_rate(pvalues, alpha):
    return float(np.mean(pvalues < alpha))

def fit_for_scenario(scenario, x, y, alpha, model_label):
    if scenario == 'exponential_lognormal' and model_label == 'log_transform':
        return fit_ols(x, np.log(y), alpha)
    return fit_ols(x, y, alpha)

def collect_replications(scenario, n, cfg, rng, beta1_generation, model_label):
    rows = []
    diagnostic = None
    for _ in range(cfg.replications):
        x, y = generate_dataset(scenario, n, rng, cfg, cfg.beta0, beta1_generation)
        fit = fit_for_scenario(scenario, x, y, cfg.alpha, model_label)
        rows.append({
            'beta0_hat': fit['beta0_hat'],
            'beta1_hat': fit['beta1_hat'],
            'ci_beta0_low': fit['ci_beta0_low'],
            'ci_beta0_high': fit['ci_beta0_high'],
            'ci_beta1_low': fit['ci_beta1_low'],
            'ci_beta1_high': fit['ci_beta1_high'],
            'pvalue_beta1': fit['pvalue_beta1'],
        })
        if diagnostic is None:
            diagnostic = {'fitted': fit['fitted'], 'resid': fit['resid']}
    return pd.DataFrame(rows), diagnostic

def save_diagnostics(fig_dir, slug, diagnostic, beta1_values):
    plt.figure(figsize=(7,4))
    plt.scatter(diagnostic['fitted'], diagnostic['resid'], alpha=0.6)
    plt.axhline(0, color='black', linestyle='--')
    plt.title(f'Residuos vs ajustados - {slug}')
    plt.tight_layout(); plt.savefig(fig_dir / f'{slug}_residuos_vs_ajustados.png', dpi=150); plt.close()

    plt.figure(figsize=(6,6))
    stats.probplot(diagnostic['resid'], dist='norm', plot=plt)
    plt.title(f'QQ-plot residuos - {slug}')
    plt.tight_layout(); plt.savefig(fig_dir / f'{slug}_qqplot_residuos.png', dpi=150); plt.close()

    plt.figure(figsize=(7,4))
    plt.hist(beta1_values, bins=30, edgecolor='black', alpha=0.85)
    plt.title(f'Histograma beta1 - {slug}')
    plt.tight_layout(); plt.savefig(fig_dir / f'{slug}_hist_beta1.png', dpi=150); plt.close()

    plt.figure(figsize=(6,4))
    plt.boxplot(beta1_values, vert=True)
    plt.title(f'Boxplot beta1 - {slug}')
    plt.tight_layout(); plt.savefig(fig_dir / f'{slug}_boxplot_beta1.png', dpi=150); plt.close()

## Cenário: C1_classico_normal (classic_normal)

Definição do cenário: erros normais clássicos, modelo linear correto.

In [ ]:
scenario = 'classic_normal'
label = SCENARIO_LABELS[scenario]
print(f'--- Cenário: {label} ({scenario}) ---')
model_labels = ['original_scale']
if scenario == 'exponential_lognormal':
    model_labels = ['original_scale', 'log_transform']
scenario_summary_rows = []
scenario_est_rows = []
rep_n = max(cfg.sample_sizes)
for n in cfg.sample_sizes:
    for model_label in model_labels:
        alt, diagnostic = collect_replications(scenario, n, cfg, rng, cfg.beta1, model_label)
        null, _ = collect_replications(scenario, n, cfg, rng, 0.0, model_label)
        size_h0 = rejection_rate(null['pvalue_beta1'].to_numpy(), cfg.alpha)
        power_h1 = rejection_rate(alt['pvalue_beta1'].to_numpy(), cfg.alpha)

        m0, b0, v0, mse0 = summarize_parameter(alt['beta0_hat'].to_numpy(), cfg.beta0)
        m1, b1, v1, mse1 = summarize_parameter(alt['beta1_hat'].to_numpy(), cfg.beta1)
        cov0 = coverage_rate(alt['ci_beta0_low'].to_numpy(), alt['ci_beta0_high'].to_numpy(), cfg.beta0)
        cov1 = coverage_rate(alt['ci_beta1_low'].to_numpy(), alt['ci_beta1_high'].to_numpy(), cfg.beta1)

        scenario_summary_rows.extend([
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta0',
             'mean_hat': m0, 'bias': b0, 'variance': v0, 'mse': mse0, 'coverage': cov0, 'size_h0': size_h0, 'power_h1': power_h1},
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta1',
             'mean_hat': m1, 'bias': b1, 'variance': v1, 'mse': mse1, 'coverage': cov1, 'size_h0': size_h0, 'power_h1': power_h1}
        ])

        temp = alt.copy()
        temp['scenario'] = scenario
        temp['scenario_label'] = label
        temp['n'] = n
        temp['model'] = model_label
        scenario_est_rows.extend(temp.to_dict(orient='records'))

        if n == rep_n:
            slug = f"{label}_n{n}_{model_label}"
            save_diagnostics(FIGURES, slug, diagnostic, alt['beta1_hat'].to_numpy())

scenario_summary_df = pd.DataFrame(scenario_summary_rows).sort_values(by=['scenario','n','model','parameter'])
scenario_est_df = pd.DataFrame(scenario_est_rows).sort_values(by=['n','model'])
compact_df = scenario_summary_df[['scenario_label','n','model','parameter','bias','variance','mse','coverage','size_h0','power_h1']]

# Salva arquivos específicos do cenario
compact_df.to_csv(TABLES / f'{scenario}_resumo_compacto.csv', index=False)
scenario_est_df.to_csv(TABLES / f'{scenario}_estimativas_brutas.csv', index=False)

compact_df.head(20)

## Cenário: C2_caudas_pesadas (heavy_tails)

Definição do cenário: erros com caudas pesadas (t-student).

In [ ]:
scenario = 'heavy_tails'
label = SCENARIO_LABELS[scenario]
print(f'--- Cenário: {label} ({scenario}) ---')
model_labels = ['original_scale']
if scenario == 'exponential_lognormal':
    model_labels = ['original_scale', 'log_transform']
scenario_summary_rows = []
scenario_est_rows = []
rep_n = max(cfg.sample_sizes)
for n in cfg.sample_sizes:
    for model_label in model_labels:
        alt, diagnostic = collect_replications(scenario, n, cfg, rng, cfg.beta1, model_label)
        null, _ = collect_replications(scenario, n, cfg, rng, 0.0, model_label)
        size_h0 = rejection_rate(null['pvalue_beta1'].to_numpy(), cfg.alpha)
        power_h1 = rejection_rate(alt['pvalue_beta1'].to_numpy(), cfg.alpha)

        m0, b0, v0, mse0 = summarize_parameter(alt['beta0_hat'].to_numpy(), cfg.beta0)
        m1, b1, v1, mse1 = summarize_parameter(alt['beta1_hat'].to_numpy(), cfg.beta1)
        cov0 = coverage_rate(alt['ci_beta0_low'].to_numpy(), alt['ci_beta0_high'].to_numpy(), cfg.beta0)
        cov1 = coverage_rate(alt['ci_beta1_low'].to_numpy(), alt['ci_beta1_high'].to_numpy(), cfg.beta1)

        scenario_summary_rows.extend([
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta0',
             'mean_hat': m0, 'bias': b0, 'variance': v0, 'mse': mse0, 'coverage': cov0, 'size_h0': size_h0, 'power_h1': power_h1},
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta1',
             'mean_hat': m1, 'bias': b1, 'variance': v1, 'mse': mse1, 'coverage': cov1, 'size_h0': size_h0, 'power_h1': power_h1}
        ])

        temp = alt.copy()
        temp['scenario'] = scenario
        temp['scenario_label'] = label
        temp['n'] = n
        temp['model'] = model_label
        scenario_est_rows.extend(temp.to_dict(orient='records'))

        if n == rep_n:
            slug = f"{label}_n{n}_{model_label}"
            save_diagnostics(FIGURES, slug, diagnostic, alt['beta1_hat'].to_numpy())

scenario_summary_df = pd.DataFrame(scenario_summary_rows).sort_values(by=['scenario','n','model','parameter'])
scenario_est_df = pd.DataFrame(scenario_est_rows).sort_values(by=['n','model'])
compact_df = scenario_summary_df[['scenario_label','n','model','parameter','bias','variance','mse','coverage','size_h0','power_h1']]

# Salva arquivos específicos do cenario
compact_df.to_csv(TABLES / f'{scenario}_resumo_compacto.csv', index=False)
scenario_est_df.to_csv(TABLES / f'{scenario}_estimativas_brutas.csv', index=False)

compact_df.head(20)

## Cenário: C3_erros_assimetricos (skewed_errors)

Definição do cenário: erros assimétricos (gamma deslocado).

In [ ]:
scenario = 'skewed_errors'
label = SCENARIO_LABELS[scenario]
print(f'--- Cenário: {label} ({scenario}) ---')
model_labels = ['original_scale']
if scenario == 'exponential_lognormal':
    model_labels = ['original_scale', 'log_transform']
scenario_summary_rows = []
scenario_est_rows = []
rep_n = max(cfg.sample_sizes)
for n in cfg.sample_sizes:
    for model_label in model_labels:
        alt, diagnostic = collect_replications(scenario, n, cfg, rng, cfg.beta1, model_label)
        null, _ = collect_replications(scenario, n, cfg, rng, 0.0, model_label)
        size_h0 = rejection_rate(null['pvalue_beta1'].to_numpy(), cfg.alpha)
        power_h1 = rejection_rate(alt['pvalue_beta1'].to_numpy(), cfg.alpha)

        m0, b0, v0, mse0 = summarize_parameter(alt['beta0_hat'].to_numpy(), cfg.beta0)
        m1, b1, v1, mse1 = summarize_parameter(alt['beta1_hat'].to_numpy(), cfg.beta1)
        cov0 = coverage_rate(alt['ci_beta0_low'].to_numpy(), alt['ci_beta0_high'].to_numpy(), cfg.beta0)
        cov1 = coverage_rate(alt['ci_beta1_low'].to_numpy(), alt['ci_beta1_high'].to_numpy(), cfg.beta1)

        scenario_summary_rows.extend([
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta0',
             'mean_hat': m0, 'bias': b0, 'variance': v0, 'mse': mse0, 'coverage': cov0, 'size_h0': size_h0, 'power_h1': power_h1},
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta1',
             'mean_hat': m1, 'bias': b1, 'variance': v1, 'mse': mse1, 'coverage': cov1, 'size_h0': size_h0, 'power_h1': power_h1}
        ])

        temp = alt.copy()
        temp['scenario'] = scenario
        temp['scenario_label'] = label
        temp['n'] = n
        temp['model'] = model_label
        scenario_est_rows.extend(temp.to_dict(orient='records'))

        if n == rep_n:
            slug = f"{label}_n{n}_{model_label}"
            save_diagnostics(FIGURES, slug, diagnostic, alt['beta1_hat'].to_numpy())

scenario_summary_df = pd.DataFrame(scenario_summary_rows).sort_values(by=['scenario','n','model','parameter'])
scenario_est_df = pd.DataFrame(scenario_est_rows).sort_values(by=['n','model'])
compact_df = scenario_summary_df[['scenario_label','n','model','parameter','bias','variance','mse','coverage','size_h0','power_h1']]

# Salva arquivos específicos do cenario
compact_df.to_csv(TABLES / f'{scenario}_resumo_compacto.csv', index=False)
scenario_est_df.to_csv(TABLES / f'{scenario}_estimativas_brutas.csv', index=False)

compact_df.head(20)

## Cenário: C4a_media_erro_nao_nula_constante (nonzero_mean_const)

Definição do cenário: média do erro diferente de zero (constante).

In [ ]:
scenario = 'nonzero_mean_const'
label = SCENARIO_LABELS[scenario]
print(f'--- Cenário: {label} ({scenario}) ---')
model_labels = ['original_scale']
if scenario == 'exponential_lognormal':
    model_labels = ['original_scale', 'log_transform']
scenario_summary_rows = []
scenario_est_rows = []
rep_n = max(cfg.sample_sizes)
for n in cfg.sample_sizes:
    for model_label in model_labels:
        alt, diagnostic = collect_replications(scenario, n, cfg, rng, cfg.beta1, model_label)
        null, _ = collect_replications(scenario, n, cfg, rng, 0.0, model_label)
        size_h0 = rejection_rate(null['pvalue_beta1'].to_numpy(), cfg.alpha)
        power_h1 = rejection_rate(alt['pvalue_beta1'].to_numpy(), cfg.alpha)

        m0, b0, v0, mse0 = summarize_parameter(alt['beta0_hat'].to_numpy(), cfg.beta0)
        m1, b1, v1, mse1 = summarize_parameter(alt['beta1_hat'].to_numpy(), cfg.beta1)
        cov0 = coverage_rate(alt['ci_beta0_low'].to_numpy(), alt['ci_beta0_high'].to_numpy(), cfg.beta0)
        cov1 = coverage_rate(alt['ci_beta1_low'].to_numpy(), alt['ci_beta1_high'].to_numpy(), cfg.beta1)

        scenario_summary_rows.extend([
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta0',
             'mean_hat': m0, 'bias': b0, 'variance': v0, 'mse': mse0, 'coverage': cov0, 'size_h0': size_h0, 'power_h1': power_h1},
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta1',
             'mean_hat': m1, 'bias': b1, 'variance': v1, 'mse': mse1, 'coverage': cov1, 'size_h0': size_h0, 'power_h1': power_h1}
        ])

        temp = alt.copy()
        temp['scenario'] = scenario
        temp['scenario_label'] = label
        temp['n'] = n
        temp['model'] = model_label
        scenario_est_rows.extend(temp.to_dict(orient='records'))

        if n == rep_n:
            slug = f"{label}_n{n}_{model_label}"
            save_diagnostics(FIGURES, slug, diagnostic, alt['beta1_hat'].to_numpy())

scenario_summary_df = pd.DataFrame(scenario_summary_rows).sort_values(by=['scenario','n','model','parameter'])
scenario_est_df = pd.DataFrame(scenario_est_rows).sort_values(by=['n','model'])
compact_df = scenario_summary_df[['scenario_label','n','model','parameter','bias','variance','mse','coverage','size_h0','power_h1']]

# Salva arquivos específicos do cenario
compact_df.to_csv(TABLES / f'{scenario}_resumo_compacto.csv', index=False)
scenario_est_df.to_csv(TABLES / f'{scenario}_estimativas_brutas.csv', index=False)

compact_df.head(20)

## Cenário: C4b_media_erro_dependente_X (nonzero_mean_x_dep)

Definição do cenário: média do erro dependente de X.

In [ ]:
scenario = 'nonzero_mean_x_dep'
label = SCENARIO_LABELS[scenario]
print(f'--- Cenário: {label} ({scenario}) ---')
model_labels = ['original_scale']
if scenario == 'exponential_lognormal':
    model_labels = ['original_scale', 'log_transform']
scenario_summary_rows = []
scenario_est_rows = []
rep_n = max(cfg.sample_sizes)
for n in cfg.sample_sizes:
    for model_label in model_labels:
        alt, diagnostic = collect_replications(scenario, n, cfg, rng, cfg.beta1, model_label)
        null, _ = collect_replications(scenario, n, cfg, rng, 0.0, model_label)
        size_h0 = rejection_rate(null['pvalue_beta1'].to_numpy(), cfg.alpha)
        power_h1 = rejection_rate(alt['pvalue_beta1'].to_numpy(), cfg.alpha)

        m0, b0, v0, mse0 = summarize_parameter(alt['beta0_hat'].to_numpy(), cfg.beta0)
        m1, b1, v1, mse1 = summarize_parameter(alt['beta1_hat'].to_numpy(), cfg.beta1)
        cov0 = coverage_rate(alt['ci_beta0_low'].to_numpy(), alt['ci_beta0_high'].to_numpy(), cfg.beta0)
        cov1 = coverage_rate(alt['ci_beta1_low'].to_numpy(), alt['ci_beta1_high'].to_numpy(), cfg.beta1)

        scenario_summary_rows.extend([
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta0',
             'mean_hat': m0, 'bias': b0, 'variance': v0, 'mse': mse0, 'coverage': cov0, 'size_h0': size_h0, 'power_h1': power_h1},
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta1',
             'mean_hat': m1, 'bias': b1, 'variance': v1, 'mse': mse1, 'coverage': cov1, 'size_h0': size_h0, 'power_h1': power_h1}
        ])

        temp = alt.copy()
        temp['scenario'] = scenario
        temp['scenario_label'] = label
        temp['n'] = n
        temp['model'] = model_label
        scenario_est_rows.extend(temp.to_dict(orient='records'))

        if n == rep_n:
            slug = f"{label}_n{n}_{model_label}"
            save_diagnostics(FIGURES, slug, diagnostic, alt['beta1_hat'].to_numpy())

scenario_summary_df = pd.DataFrame(scenario_summary_rows).sort_values(by=['scenario','n','model','parameter'])
scenario_est_df = pd.DataFrame(scenario_est_rows).sort_values(by=['n','model'])
compact_df = scenario_summary_df[['scenario_label','n','model','parameter','bias','variance','mse','coverage','size_h0','power_h1']]

# Salva arquivos específicos do cenario
compact_df.to_csv(TABLES / f'{scenario}_resumo_compacto.csv', index=False)
scenario_est_df.to_csv(TABLES / f'{scenario}_estimativas_brutas.csv', index=False)

compact_df.head(20)

## Cenário: C5_erros_correlacionados_ar1 (ar1_errors)

Definição do cenário: erros correlacionados AR(1).

In [ ]:
scenario = 'ar1_errors'
label = SCENARIO_LABELS[scenario]
print(f'--- Cenário: {label} ({scenario}) ---')
model_labels = ['original_scale']
if scenario == 'exponential_lognormal':
    model_labels = ['original_scale', 'log_transform']
scenario_summary_rows = []
scenario_est_rows = []
rep_n = max(cfg.sample_sizes)
for n in cfg.sample_sizes:
    for model_label in model_labels:
        alt, diagnostic = collect_replications(scenario, n, cfg, rng, cfg.beta1, model_label)
        null, _ = collect_replications(scenario, n, cfg, rng, 0.0, model_label)
        size_h0 = rejection_rate(null['pvalue_beta1'].to_numpy(), cfg.alpha)
        power_h1 = rejection_rate(alt['pvalue_beta1'].to_numpy(), cfg.alpha)

        m0, b0, v0, mse0 = summarize_parameter(alt['beta0_hat'].to_numpy(), cfg.beta0)
        m1, b1, v1, mse1 = summarize_parameter(alt['beta1_hat'].to_numpy(), cfg.beta1)
        cov0 = coverage_rate(alt['ci_beta0_low'].to_numpy(), alt['ci_beta0_high'].to_numpy(), cfg.beta0)
        cov1 = coverage_rate(alt['ci_beta1_low'].to_numpy(), alt['ci_beta1_high'].to_numpy(), cfg.beta1)

        scenario_summary_rows.extend([
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta0',
             'mean_hat': m0, 'bias': b0, 'variance': v0, 'mse': mse0, 'coverage': cov0, 'size_h0': size_h0, 'power_h1': power_h1},
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta1',
             'mean_hat': m1, 'bias': b1, 'variance': v1, 'mse': mse1, 'coverage': cov1, 'size_h0': size_h0, 'power_h1': power_h1}
        ])

        temp = alt.copy()
        temp['scenario'] = scenario
        temp['scenario_label'] = label
        temp['n'] = n
        temp['model'] = model_label
        scenario_est_rows.extend(temp.to_dict(orient='records'))

        if n == rep_n:
            slug = f"{label}_n{n}_{model_label}"
            save_diagnostics(FIGURES, slug, diagnostic, alt['beta1_hat'].to_numpy())

scenario_summary_df = pd.DataFrame(scenario_summary_rows).sort_values(by=['scenario','n','model','parameter'])
scenario_est_df = pd.DataFrame(scenario_est_rows).sort_values(by=['n','model'])
compact_df = scenario_summary_df[['scenario_label','n','model','parameter','bias','variance','mse','coverage','size_h0','power_h1']]

# Salva arquivos específicos do cenario
compact_df.to_csv(TABLES / f'{scenario}_resumo_compacto.csv', index=False)
scenario_est_df.to_csv(TABLES / f'{scenario}_estimativas_brutas.csv', index=False)

compact_df.head(20)

## Cenário: C6_relacao_exponencial_lognormal (exponential_lognormal)

Definição do cenário: relação exponencial com ruído log-normal; aqui comparamos escala original e transformação `log(Y)`.

In [ ]:
scenario = 'exponential_lognormal'
label = SCENARIO_LABELS[scenario]
print(f'--- Cenário: {label} ({scenario}) ---')
model_labels = ['original_scale', 'log_transform']
scenario_summary_rows = []
scenario_est_rows = []
rep_n = max(cfg.sample_sizes)
for n in cfg.sample_sizes:
    for model_label in model_labels:
        alt, diagnostic = collect_replications(scenario, n, cfg, rng, cfg.beta1, model_label)
        null, _ = collect_replications(scenario, n, cfg, rng, 0.0, model_label)
        size_h0 = rejection_rate(null['pvalue_beta1'].to_numpy(), cfg.alpha)
        power_h1 = rejection_rate(alt['pvalue_beta1'].to_numpy(), cfg.alpha)

        m0, b0, v0, mse0 = summarize_parameter(alt['beta0_hat'].to_numpy(), cfg.beta0)
        m1, b1, v1, mse1 = summarize_parameter(alt['beta1_hat'].to_numpy(), cfg.beta1)
        cov0 = coverage_rate(alt['ci_beta0_low'].to_numpy(), alt['ci_beta0_high'].to_numpy(), cfg.beta0)
        cov1 = coverage_rate(alt['ci_beta1_low'].to_numpy(), alt['ci_beta1_high'].to_numpy(), cfg.beta1)

        scenario_summary_rows.extend([
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta0',
             'mean_hat': m0, 'bias': b0, 'variance': v0, 'mse': mse0, 'coverage': cov0, 'size_h0': size_h0, 'power_h1': power_h1},
            {'scenario': scenario, 'scenario_label': label, 'n': n, 'model': model_label, 'parameter': 'beta1',
             'mean_hat': m1, 'bias': b1, 'variance': v1, 'mse': mse1, 'coverage': cov1, 'size_h0': size_h0, 'power_h1': power_h1}
        ])

        temp = alt.copy()
        temp['scenario'] = scenario
        temp['scenario_label'] = label
        temp['n'] = n
        temp['model'] = model_label
        scenario_est_rows.extend(temp.to_dict(orient='records'))

        if n == rep_n:
            slug = f"{label}_n{n}_{model_label}"
            save_diagnostics(FIGURES, slug, diagnostic, alt['beta1_hat'].to_numpy())

scenario_summary_df = pd.DataFrame(scenario_summary_rows).sort_values(by=['scenario','n','model','parameter'])
scenario_est_df = pd.DataFrame(scenario_est_rows).sort_values(by=['n','model'])
compact_df = scenario_summary_df[['scenario_label','n','model','parameter','bias','variance','mse','coverage','size_h0','power_h1']]

# Salva arquivos específicos do cenario
compact_df.to_csv(TABLES / f'{scenario}_resumo_compacto.csv', index=False)
scenario_est_df.to_csv(TABLES / f'{scenario}_estimativas_brutas.csv', index=False)

compact_df.head(20)